# Feature Selection Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Generate synthetic data with known feature structure

In [ ]:
```python

import numpy as np

def make_feature_selection_data(n_samples=500, seed=42):

    rng = np.random.RandomState(seed)

    x1 = rng.randn(n_samples)

    x2 = rng.randn(n_samples)

    x3 = rng.randn(n_samples)

    x4 = x1 + 0.1 * rng.randn(n_samples)

    x5 = x2 + 0.1 * rng.randn(n_samples)

    informative = np.column_stack([x1, x2, x3, x4, x5])

    correlated = np.column_stack([

        x1 * 0.9 + 0.1 * rng.randn(n_samples),

        x2 * 0.8 + 0.2 * rng.randn(n_samples),

        x3 * 0.7 + 0.3 * rng.randn(n_samples),

        x1 * 0.5 + x2 * 0.5 + 0.1 * rng.randn(n_samples),

        x2 * 0.6 + x3 * 0.4 + 0.1 * rng.randn(n_samples),

    ])

    noise = rng.randn(n_samples, 10) * 0.5

    X = np.hstack([informative, correlated, noise])

    y = (2 * x1 - 1.5 * x2 + x3 + 0.5 * rng.randn(n_samples) > 0).astype(int)

    feature_names = (

        [f"info_{i}" for i in range(5)]

        + [f"corr_{i}" for i in range(5)]

        + [f"noise_{i}" for i in range(10)]

    )

    return X, y, feature_names

In [ ]:
```

We know the ground truth: features 0-4 are informative (plus 3 and 4 are correlated copies of 0 and 1), features 5-9 are correlated with informative features, features 10-19 are pure noise. A good selection method should rank 0-4 highest and 10-19 lowest.

### Step 2: Variance threshold

In [ ]:
```python

def variance_threshold(X, threshold=0.01):

    variances = np.var(X, axis=0)

    mask = variances > threshold

    return mask, variances

In [ ]:
```

### Step 3: Mutual information (discrete)

In [ ]:
```python

def discretize(x, n_bins=10):

    min_val, max_val = x.min(), x.max()

    if max_val == min_val:

        return np.zeros_like(x, dtype=int)

    bin_edges = np.linspace(min_val, max_val, n_bins + 1)

    binned = np.digitize(x, bin_edges[1:-1])

    return binned

def mutual_information(X, y, n_bins=10):

    n_samples, n_features = X.shape

    mi_scores = np.zeros(n_features)

    y_vals, y_counts = np.unique(y, return_counts=True)

    p_y = y_counts / n_samples

    for f in range(n_features):

        x_binned = discretize(X[:, f], n_bins)

        x_vals, x_counts = np.unique(x_binned, return_counts=True)

        p_x = dict(zip(x_vals, x_counts / n_samples))

        mi = 0.0

        for xv in x_vals:

            for yi, yv in enumerate(y_vals):

                joint_mask = (x_binned == xv) & (y == yv)

                p_xy = np.sum(joint_mask) / n_samples

                if p_xy > 0:

                    mi += p_xy * np.log(p_xy / (p_x[xv] * p_y[yi]))

        mi_scores[f] = mi

    return mi_scores

In [ ]:
```

### Step 4: Recursive Feature Elimination

In [ ]:
```python

def simple_logistic_importance(X, y, lr=0.1, epochs=100):

    n_samples, n_features = X.shape

    w = np.zeros(n_features)

    b = 0.0

    for _ in range(epochs):

        z = X @ w + b

        pred = 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

        error = pred - y

        w -= lr * (X.T @ error) / n_samples

        b -= lr * np.mean(error)

    return w, b

def rfe(X, y, n_features_to_select=5, lr=0.1, epochs=100):

    n_total = X.shape[1]

    remaining = list(range(n_total))

    rankings = np.ones(n_total, dtype=int)

    rank = n_total

    while len(remaining) > n_features_to_select:

        X_subset = X[:, remaining]

        w, _ = simple_logistic_importance(X_subset, y, lr, epochs)

        importances = np.abs(w)

        least_idx = np.argmin(importances)

        original_idx = remaining[least_idx]

        rankings[original_idx] = rank

        rank -= 1

        remaining.pop(least_idx)

    for idx in remaining:

        rankings[idx] = 1

    selected_mask = rankings == 1

    return selected_mask, rankings

In [ ]:
```

### Step 5: L1 feature selection

In [ ]:
```python

def soft_threshold(w, alpha):

    return np.sign(w) * np.maximum(np.abs(w) - alpha, 0)

def l1_feature_selection(X, y, alpha=0.1, lr=0.01, epochs=500):

    n_samples, n_features = X.shape

    w = np.zeros(n_features)

    b = 0.0

    for _ in range(epochs):

        z = X @ w + b

        pred = 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

        error = pred - y

        gradient_w = (X.T @ error) / n_samples

        gradient_b = np.mean(error)

        w -= lr * gradient_w

        w = soft_threshold(w, lr * alpha)

        b -= lr * gradient_b

    selected_mask = np.abs(w) > 1e-6

    return selected_mask, w

In [ ]:
```

### Step 6: Tree-based importance (simple decision tree)

In [ ]:
```python

def gini_impurity(y):

    if len(y) == 0:

        return 0.0

    classes, counts = np.unique(y, return_counts=True)

    probs = counts / len(y)

    return 1.0 - np.sum(probs ** 2)

def best_split(X, y, feature_idx):

    values = np.unique(X[:, feature_idx])

    if len(values) <= 1:

        return None, -1.0

    best_threshold = None

    best_gain = -1.0

    parent_gini = gini_impurity(y)

    n = len(y)

    for i in range(len(values) - 1):

        threshold = (values[i] + values[i + 1]) / 2.0

        left_mask = X[:, feature_idx] <= threshold

        right_mask = ~left_mask

        n_left = np.sum(left_mask)

        n_right = np.sum(right_mask)

        if n_left == 0 or n_right == 0:

            continue

        gain = parent_gini - (n_left / n) * gini_impurity(y[left_mask]) - (n_right / n) * gini_impurity(y[right_mask])

        if gain > best_gain:

            best_gain = gain

            best_threshold = threshold

    return best_threshold, best_gain

def tree_importance(X, y, n_trees=50, max_depth=5, seed=42):

    rng = np.random.RandomState(seed)

    n_samples, n_features = X.shape

    importances = np.zeros(n_features)

    for _ in range(n_trees):

        sample_idx = rng.choice(n_samples, size=n_samples, replace=True)

        feature_subset = rng.choice(n_features, size=max(1, int(np.sqrt(n_features))), replace=False)

        X_boot = X[sample_idx]

        y_boot = y[sample_idx]

        tree_imp = _build_tree_importance(X_boot, y_boot, feature_subset, max_depth)

        importances += tree_imp

    total = importances.sum()

    if total > 0:

        importances /= total

    return importances

def _build_tree_importance(X, y, feature_subset, max_depth, depth=0):

    n_features = X.shape[1]

    importances = np.zeros(n_features)

    if depth >= max_depth or len(np.unique(y)) <= 1 or len(y) < 4:

        return importances

    best_feature = None

    best_threshold = None

    best_gain = -1.0

    for f in feature_subset:

        threshold, gain = best_split(X, y, f)

        if gain > best_gain:

            best_gain = gain

            best_feature = f

            best_threshold = threshold

    if best_feature is None or best_gain <= 0:

        return importances

    importances[best_feature] += best_gain * len(y)

    left_mask = X[:, best_feature] <= best_threshold

    right_mask = ~left_mask

    importances += _build_tree_importance(X[left_mask], y[left_mask], feature_subset, max_depth, depth + 1)

    importances += _build_tree_importance(X[right_mask], y[right_mask], feature_subset, max_depth, depth + 1)

    return importances

In [ ]:
```

### Step 7: Run all methods and compare

The code file runs all five methods on the same synthetic dataset and prints a comparison table showing which features each method selects.

## Exercises

In [ ]:
1. **Forward selection**: implement the opposite of RFE. Start with zero features. At each step, add the feature that improves model performance the most. Stop when adding features no longer helps. Compare the selected features against RFE results. Which is faster? Which gives better results?

2. **Stability selection**: run L1 feature selection 50 times, each time on a random 80% subsample of the data, with slightly different alpha values. Count how often each feature is selected. Features selected in > 80% of runs are "stable." Compare stable features against single-run L1 selection. Which is more reliable?

3. **Multicollinearity detection**: compute the correlation matrix for all features. Implement a function that, given a correlation threshold (e.g., 0.9), removes one feature from each highly-correlated pair (keeping the one with higher mutual information with the target). Test on the synthetic dataset and verify it removes the redundant correlated features.

4. **Feature selection pipeline**: chain variance threshold, mutual information filter, and RFE into a single pipeline. First remove near-zero-variance features, then keep the top 50% by mutual information, then run RFE on the survivors. Compare this pipeline against running RFE alone on all features. Is the pipeline faster? Is it equally accurate?

5. **Permutation importance from scratch**: implement permutation importance. For each feature, shuffle its values 10 times, measure the average drop in F1 score. Compare the ranking against tree-based importance. Find cases where they disagree and explain why (hint: correlated features).